In [2]:
import pandas as pd
import re
import os


In [ ]:
# FHIR clinical context
 #       ↓
#ImagingStudy

In [15]:
imaging_studies = pd.read_csv(
    "synthetic_imaging_studies.csv"
)

In [10]:
reports = pd.read_csv(
    "synthetic_radiology_reports.csv"
)

In [11]:
reports.head()

,report_id,patient_id,accession_number,service_request_id,encounter_id,report_datetime,modality,exam_name,report_text
0,RPT-2001,PAT-001,ACC-2026-00142,SR-1042,ENC-501,2026-05-14 10:48:00,CT,CT CHEST WITH IV CONTRAST,\nEXAM:\nCT CHEST WITH IV CONTRAST\n\nCLINICAL...
1,RPT-2002,PAT-001,ACC-2026-00143,SR-1043,ENC-501,2026-05-14 12:32:00,CT,CT ABDOMEN AND PELVIS WITH IV CONTRAST,\nEXAM:\nCT ABDOMEN AND PELVIS WITH IV CONTRAS...
2,RPT-2003,PAT-001,ACC-2026-00144,SR-1044,ENC-501,2026-05-14 14:51:00,DX,CHEST RADIOGRAPH 2 VIEWS,"\nEXAM:\nCHEST RADIOGRAPH, PA AND LATERAL\n\nC..."
3,RPT-2004,PAT-001,ACC-2026-00145,SR-1045,ENC-502,2026-05-16 09:05:00,MG,DIAGNOSTIC BILATERAL MAMMOGRAPHY,\nEXAM:\nBILATERAL DIAGNOSTIC DIGITAL MAMMOGRA...
4,RPT-1990,PAT-001,ACC-2025-00891,SR-0891,ENC-430,2025-11-10 16:22:00,CT,CT CHEST WITHOUT IV CONTRAST,\nEXAM:\nCT CHEST WITHOUT IV CONTRAST\n\nCLINI...


In [16]:
selected_study = (
    imaging_studies[
        imaging_studies["accession_number"] == "ACC-2026-00142"
    ]
    .iloc[0]
    .to_dict()
)

selected_study

{'patient_id': 'PAT-001',
 'study_uid': '1.2.840.113619.2.55.3.1001',
 'accession_number': 'ACC-2026-00142',
 'service_request_id': 'SR-1042',
 'encounter_id': 'ENC-501',
 'study_datetime': '2026-05-14 09:32:00',
 'modality': 'CT',
 'description': 'CT CHEST WITH IV CONTRAST'}

In [ ]:
## Now the notebook can present the task without revealing the answer:

In [17]:
print("Selected ImagingStudy")
print("-" * 50)

for key, value in selected_study.items():
    print(f"{key:22s}: {value}")

Selected ImagingStudy
--------------------------------------------------
patient_id            : PAT-001
study_uid             : 1.2.840.113619.2.55.3.1001
accession_number      : ACC-2026-00142
service_request_id    : SR-1042
encounter_id          : ENC-501
study_datetime        : 2026-05-14 09:32:00
modality              : CT
description           : CT CHEST WITH IV CONTRAST


In [18]:
candidate_reports = reports[
    reports["patient_id"] ==
    selected_study["patient_id"]
].copy()

candidate_reports[
    [
        "report_id",
        "report_datetime",
        "modality",
        "exam_name",
        "accession_number",
        "service_request_id",
        "encounter_id"
    ]
]

,report_id,report_datetime,modality,exam_name,accession_number,service_request_id,encounter_id
0,RPT-2001,2026-05-14 10:48:00,CT,CT CHEST WITH IV CONTRAST,ACC-2026-00142,SR-1042,ENC-501
1,RPT-2002,2026-05-14 12:32:00,CT,CT ABDOMEN AND PELVIS WITH IV CONTRAST,ACC-2026-00143,SR-1043,ENC-501
2,RPT-2003,2026-05-14 14:51:00,DX,CHEST RADIOGRAPH 2 VIEWS,ACC-2026-00144,SR-1044,ENC-501
3,RPT-2004,2026-05-16 09:05:00,MG,DIAGNOSTIC BILATERAL MAMMOGRAPHY,ACC-2026-00145,SR-1045,ENC-502
4,RPT-1990,2025-11-10 16:22:00,CT,CT CHEST WITHOUT IV CONTRAST,ACC-2025-00891,SR-0891,ENC-430
5,RPT-2005,2026-05-14 07:45:00,CT,CT PULMONARY ANGIOGRAM,ACC-2026-00140,SR-1040,ENC-501
6,RPT-2006,2026-05-14 09:15:00,DX,PORTABLE CHEST RADIOGRAPH,ACC-2026-00141,SR-1041,ENC-501
7,RPT-2007,2026-05-20 13:12:00,MG,LEFT DIAGNOSTIC MAMMOGRAPHY,ACC-2026-00150,SR-1050,ENC-503
8,RPT-2008,2026-06-03 11:35:00,CT,CT ABDOMEN WITH IV CONTRAST,ACC-2026-00201,SR-1101,ENC-510
9,RPT-2009,2026-05-28 10:10:00,CT,CT CHEST WITHOUT IV CONTRAST,NaN,SR-1099,ENC-509


In [ ]:
## Hands-on Exercise: Which radiology report corresponds to the selected ImagingStudy?  

#Consider the patient, accession number, ServiceRequest/order, encounter, modality, and temporal relationship. Do not use report-text similarity alone.

In [ ]:
#ambiguity/abstention

In [20]:


def evaluate_report_matches(study, reports):
    """
    Compare candidate radiology reports against a selected ImagingStudy.
    """

    candidates = reports[
        reports["patient_id"] == study["patient_id"]
    ].copy()

    # Convert times
    study_time = pd.to_datetime(study["study_datetime"])

    candidates["report_datetime"] = pd.to_datetime(
        candidates["report_datetime"],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Matching criteria
    # --------------------------------------------------------

    candidates["accession_match"] = (
        candidates["accession_number"].fillna("").astype(str)
        ==
        str(study["accession_number"])
    )

    candidates["order_match"] = (
        candidates["service_request_id"].fillna("").astype(str)
        ==
        str(study["service_request_id"])
    )

    candidates["encounter_match"] = (
        candidates["encounter_id"].fillna("").astype(str)
        ==
        str(study["encounter_id"])
    )

    candidates["modality_match"] = (
        candidates["modality"].fillna("").astype(str)
        ==
        str(study["modality"])
    )

    # --------------------------------------------------------
    # Temporal distance
    # --------------------------------------------------------

    candidates["time_difference_hours"] = (
        candidates["report_datetime"] - study_time
    ).abs().dt.total_seconds() / 3600

    # --------------------------------------------------------
    # Define the high-confidence match
    # --------------------------------------------------------

    candidates["matched"] = (
        candidates["accession_match"]
        &
        candidates["order_match"]
    )

    # Sort best candidates first
    candidates = candidates.sort_values(
        by=[
            "matched",
            "accession_match",
            "order_match",
            "encounter_match",
            "modality_match",
            "time_difference_hours"
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
            True
        ]
    )

    return candidates

In [22]:
match_results = evaluate_report_matches(
    selected_study,
    candidate_reports
)

In [23]:
display_cols = [
    "report_id",
    "exam_name",
    "accession_match",
    "order_match",
    "encounter_match",
    "modality_match",
    "time_difference_hours",
    "matched"
]

(
    match_results[display_cols]
    .style
    .apply(
        lambda row:
            ["background-color: #d4edda"] * len(row)
            if row["matched"]
            else ["background-color: #f8d7da"] * len(row),
        axis=1
    )
    .format({
        "time_difference_hours": "{:.1f}"
    })
)

,report_id,exam_name,accession_match,order_match,encounter_match,modality_match,time_difference_hours,matched
0,RPT-2001,CT CHEST WITH IV CONTRAST,True,True,True,True,1.3,True
5,RPT-2005,CT PULMONARY ANGIOGRAM,False,False,True,True,1.8,False
1,RPT-2002,CT ABDOMEN AND PELVIS WITH IV CONTRAST,False,False,True,True,3.0,False
6,RPT-2006,PORTABLE CHEST RADIOGRAPH,False,False,True,False,0.3,False
2,RPT-2003,CHEST RADIOGRAPH 2 VIEWS,False,False,True,False,5.3,False
9,RPT-2009,CT CHEST WITHOUT IV CONTRAST,False,False,False,True,336.6,False
8,RPT-2008,CT ABDOMEN WITH IV CONTRAST,False,False,False,True,482.1,False
4,RPT-1990,CT CHEST WITHOUT IV CONTRAST,False,False,False,True,4433.2,False
3,RPT-2004,DIAGNOSTIC BILATERAL MAMMOGRAPHY,False,False,False,False,47.5,False
7,RPT-2007,LEFT DIAGNOSTIC MAMMOGRAPHY,False,False,False,False,147.7,False


In [24]:
def classify_match(row):

    if row["matched"]:
        return "MATCH"

    elif row["encounter_match"] and row["modality_match"]:
        return "POSSIBLE DISTRACTOR"

    else:
        return "NON-MATCH"


match_results["match_status"] = match_results.apply(
    classify_match,
    axis=1
)

In [28]:



def extract_report_section(text, section="FINDINGS"):
    """
    Extract a section from a radiology report.

    Parameters
    ----------
    text : str
        Full radiology report.

    section : str
        Section to extract, e.g. FINDINGS, IMPRESSION,
        HISTORY, TECHNIQUE.

    Returns
    -------
    str or None
    """

    if pd.isna(text):
        return None

    text = str(text)

    # Match requested section and stop at the next
    # uppercase section heading or end of report
    pattern = rf"""
        (?:^|\n)\s*{re.escape(section)}\s*:?\s*
        (.*?)
        (?=
            \n\s*[A-Z][A-Z0-9 /&()_-]{{2,}}\s*:?\s*\n
            |
            \Z
        )
    """

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE | re.DOTALL | re.VERBOSE
    )

    if match:
        return match.group(1).strip()

    return None

In [29]:
matched_report = match_results[
    match_results["matched"]
].iloc[0]

findings = extract_report_section(
    matched_report["report_text"],
    section="FINDINGS"
)

print(findings)

LUNGS AND AIRWAYS:
8 mm solid pulmonary nodule in the posterior right upper
lobe, previously measuring 6 mm. Mild bibasilar dependent
atelectatic change. No focal airspace consolidation.


In [30]:
history = extract_report_section(
    matched_report["report_text"],
    "CLINICAL HISTORY"
)

technique = extract_report_section(
    matched_report["report_text"],
    "TECHNIQUE"
)

impression = extract_report_section(
    matched_report["report_text"],
    "IMPRESSION"
)

In [31]:
history = extract_report_section(
    matched_report["report_text"],
    "CLINICAL HISTORY"
)

technique = extract_report_section(
    matched_report["report_text"],
    "TECHNIQUE"
)

impression = extract_report_section(
    matched_report["report_text"],
    "IMPRESSION"
)

In [32]:
sections = {}

for section in [
    "CLINICAL HISTORY",
    "COMPARISON",
    "TECHNIQUE",
    "FINDINGS",
    "IMPRESSION"
]:
    sections[section] = extract_report_section(
        matched_report["report_text"],
        section
    )

sections

{'CLINICAL HISTORY': 'Persistent cough. History of smoking. Pulmonary nodule\nidentified on prior examination.',
 'COMPARISON': 'CT chest dated 11/10/2025.',
 'TECHNIQUE': 'Contrast-enhanced CT images of the chest were obtained from\nthe thoracic inlet through the upper abdomen.',
 'FINDINGS': 'LUNGS AND AIRWAYS:\n8 mm solid pulmonary nodule in the posterior right upper\nlobe, previously measuring 6 mm. Mild bibasilar dependent\natelectatic change. No focal airspace consolidation.',
 'IMPRESSION': '1. Interval increase in size of an 8 mm right upper lobe\n   pulmonary nodule.\n2. No thoracic lymphadenopathy.\n3. Indeterminate hepatic lesion. Please refer to the\n   separately performed CT abdomen and pelvis.'}

In [33]:
sections = {
    section: extract_report_section(matched_report["report_text"], section)
    for section in [
        "CLINICAL HISTORY",
        "COMPARISON",
        "TECHNIQUE",
        "FINDINGS",
        "IMPRESSION"
    ]
}

In [ ]:
## Which fields should actually become the VLM target?

## For image interpretation, FINDINGS may be useful. For impression generation, IMPRESSION is the target. CLINICAL HISTORY may be useful context but can also introduce information unavailable from the image.

In [34]:
def clean_report_text(text):
    if pd.isna(text):
        return None

    text = html.unescape(str(text))
    text = text.replace("\xa0", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [35]:
training_example = {
    "patient_id": selected_study["patient_id"],
    "study_uid": selected_study["study_uid"],
    "accession_number": selected_study["accession_number"],
    "report_id": matched_report["report_id"],
    "service_request_id": selected_study["service_request_id"],
    "modality": selected_study["modality"],
    "study_datetime": selected_study["study_datetime"],
    "match_method": "accession+service_request",
    "match_status": "HIGH_CONFIDENCE",
    "findings": findings
}

In [36]:
training_example

{'patient_id': 'PAT-001',
 'study_uid': '1.2.840.113619.2.55.3.1001',
 'accession_number': 'ACC-2026-00142',
 'report_id': 'RPT-2001',
 'service_request_id': 'SR-1042',
 'modality': 'CT',
 'study_datetime': '2026-05-14 09:32:00',
 'match_method': 'accession+service_request',
 'match_status': 'HIGH_CONFIDENCE',
 'findings': 'LUNGS AND AIRWAYS:\n8 mm solid pulmonary nodule in the posterior right upper\nlobe, previously measuring 6 mm. Mild bibasilar dependent\natelectatic change. No focal airspace consolidation.'}